# PneumoScan AI — Evaluate trained models (Google Colab)

Evaluates already-trained models: test-set metrics, Grad-CAM analysis and optional external validation on the RSNA challenge data (needs a free Kaggle account).

**Before you start:** *Runtime → Change runtime type → T4 GPU* (free), then *Runtime → Run all*,
or run the cells one by one with **Shift + Enter**. Full explanation: `docs/guide/15-train-from-scratch-step-by-step.md`.

> Data use: the Kermany dataset is public (CC BY 4.0). Testing with real patient images requires consent and
> appropriate institutional approval.

## 1. Get the code
Change `REPO_URL` if you use your own fork.

In [ ]:
import os
REPO_URL = "https://github.com/UzumakiAnirudh/Pneumonia-Check.git"   # <- your repository
if not os.path.exists("/content/project"):
    !git clone -q $REPO_URL /content/project
%cd /content/project/training
!pip -q install -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Download the Kermany dataset (1.2 GB, no account needed)
From the original Mendeley Data release — takes 2–5 minutes.

In [ ]:
!python download_data.py kermany

## 3. Patient-grouped 70/15/15 split
Pools all images, removes 32 exact duplicates, keeps every patient in a single split. Expect **5,824 images, 2,789 patients**.

In [ ]:
!python prepare_data.py --data-root data/raw/chest_xray --out data/splits.csv

## 4. Upload trained weights
Upload `pneumoscan_trained.zip` from the training notebook (or skip if you trained in this session).

In [ ]:
# from google.colab import files; files.upload()
# !cd /content/project && unzip -o /content/pneumoscan_trained.zip
!ls -la ../backend/weights

## 5. Internal test-set evaluation

In [ ]:
!python evaluate.py --workers 2

In [ ]:
from IPython.display import Image, display
import glob
for f in sorted(glob.glob("outputs/figures/*.png"))[:12]:
    print(f); display(Image(f, width=320))

## 6. Grad-CAM analysis

In [ ]:
for arch in ["densenet", "swin"]:
    !python gradcam_analysis.py --arch $arch --per-category 6 --max-images 400

## 7. Optional — external validation on RSNA (adults)
Upload your `kaggle.json` (Kaggle → Settings → API → *Create New Token*) and accept the competition rules on kaggle.com first.

In [ ]:
# from google.colab import files; files.upload()          # kaggle.json
# !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle competitions download -q -c rsna-pneumonia-detection-challenge -p /content/rsna
# !unzip -q -o /content/rsna/rsna-pneumonia-detection-challenge.zip -d /content/rsna
# !python external_validate.py --name "RSNA Pneumonia Detection" --rsna-labels /content/rsna/stage_2_train_labels.csv --rsna-images /content/rsna/stage_2_train_images --limit 3000

## 8. Download results

In [ ]:
!cd /content/project && zip -qr /content/pneumoscan_metrics.zip backend/metrics training/outputs
from google.colab import files
files.download("/content/pneumoscan_metrics.zip")